In [1]:
file_path = "../Data/SMSSpamCollection"

with open(file_path, "r", encoding="utf-8") as file:
    for i in range(5):
        print(file.readline().strip())

ham	Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
ham	Ok lar... Joking wif u oni...
spam	Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
ham	U dun say so early hor... U c already then say...
ham	Nah I don't think he goes to usf, he lives around here though


In [2]:
import pandas as pd

# Load the SMS Spam Collection dataset
spam_df = pd.read_csv(
    "../Data/SMSSpamCollection",
    sep="\t",
    header=None,
    names=["label", "message"]
)

# Display the first 5 rows
display(spam_df.head())

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
print("Dataset Shape:", spam_df.shape)

print("\nDataset Information:")
print(spam_df.info())

print("\nMissing Values:")
print(spam_df.isnull().sum())

print("\nLabel Distribution:")
print(spam_df["label"].value_counts())

Dataset Shape: (5572, 2)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB
None

Missing Values:
label      0
message    0
dtype: int64

Label Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


In [4]:
spam_df["label_num"] = spam_df["label"].map({
    "ham": 0,
    "spam": 1
})

display(spam_df.head())

print("\nLabel Mapping:")
print(spam_df["label_num"].value_counts())

,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0



Label Mapping:
label_num
0    4825
1     747
Name: count, dtype: int64


In [5]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

spam_df["clean_message"] = spam_df["message"].apply(clean_text)

display(spam_df[["message", "clean_message"]].head())

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in 2 a wkly comp to win fa cup fina...
3,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


In [6]:
from sklearn.model_selection import train_test_split

X = spam_df["clean_message"]

y = spam_df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data :", X_test.shape)
print("Training Labels:", y_train.shape)
print("Testing Labels :", y_test.shape)

Training Data: (4457,)
Testing Data : (1115,)
Training Labels: (4457,)
Testing Labels : (1115,)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")

X_train_tfidf = vectorizer.fit_transform(X_train)

X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF Shape:", X_train_tfidf.shape)
print("Testing TF-IDF Shape :", X_test_tfidf.shape)

Training TF-IDF Shape: (4457, 8094)
Testing TF-IDF Shape : (1115, 8094)


In [8]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()

model.fit(X_train_tfidf, y_train)

print("Naive Bayes model trained successfully!")

Naive Bayes model trained successfully!


In [9]:
y_pred = model.predict(X_test_tfidf)

print("First 10 Predictions:")
print(y_pred[:10])

print("\nActual Labels:")
print(y_test.iloc[:10].values)

First 10 Predictions:
[0 0 0 1 0 0 0 0 0 0]

Actual Labels:
[0 0 0 1 0 0 0 0 0 0]


In [10]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9632286995515695

Confusion Matrix:
[[966   0]
 [ 41 108]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [11]:
import joblib

# Save the Naive Bayes spam classifier
joblib.dump(model, "../Models/spam_classifier.pkl")

# Save the TF-IDF vectorizer
joblib.dump(vectorizer, "../Models/spam_tfidf_vectorizer.pkl")

print("Spam classifier and TF-IDF vectorizer saved successfully!")

Spam classifier and TF-IDF vectorizer saved successfully!


In [12]:
spam_df.to_csv("../Data/spam_dataset_processed.csv", index=False)

print("Processed spam dataset saved successfully!")

Processed spam dataset saved successfully!
